In [41]:
import time
import urllib.request
import re
import requests
import pandas as pd
import os

from selenium import webdriver
from selenium.webdriver import ActionChains
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.keys import Keys
from bs4 import BeautifulSoup
from selenium.common.exceptions import TimeoutException


In [ ]:
def yes24_crawl(max_page): # max_page, 
    
    search_list = []
    title_list = []
    name_list = []
    price_list = []
    com_list = []
    date_list = []
    image_list = []

    folder_path = r"imgaes\yes24"

    url = "https://www.yes24.com/Main/default.aspx"

    driver = webdriver.Chrome()
    driver.maximize_window()

    wait = WebDriverWait(driver, 10)

    driver.get(url)
    ban = True
    need_input = True

    while ban:
        if need_input:
            keyword = input("검색어 입력: ")
            need_input = False

        try:
            search_box = wait.until(
                EC.presence_of_element_located((By.ID, "query"))
            )

            search_box.clear()
            search_box.send_keys(keyword)
            search_box.send_keys(Keys.ENTER)

            try:
                WebDriverWait(driver, 3).until(EC.alert_is_present())
                alert = driver.switch_to.alert
                alert.accept()
                    
                time.sleep(2)
                need_input = True
                continue 
                    
            except TimeoutException:
                pass
            ban = False
        
        except Exception as e:
            continue
    # ----------------------------------------------------- 검색

    for page in range(1, max_page + 1): 
            
            if page == max_page + 1:                      
                print(f"크롤링 끝")
                break
            

            print(f"{page}번 페이지 크롤링")

            req = driver.page_source

            soup = BeautifulSoup(req, "html.parser")
            
            ALL_books = soup.find("ul", id = "yesSchList").find_all("li")

            for row in ALL_books:
                if not row.select_one(".gd_name"): 
                    continue
                
                title = row.select_one(".gd_name").text 
                name = row.select_one(".info_auth").text if row.select_one(".info_auth") else ""
                price = row.select_one(".yes_b").text if row.select_one(".yes_b") else ""
                com = row.select_one(".info_pub").text if row.select_one(".info_pub") else ""
                ddatee = row.select_one(".info_date").text.strip() if row.select_one(".info_date") else ""
                img_tag = row.select_one(".img_bdr img")
                

                if img_tag:
                    img_url = img_tag.get("data-original") or img_tag.get("src")
                    
                    if img_url:
                        try: 
                            safe_title = re.sub(r'[\/:*?"<>|]', '', title) 

                            img_save = os.path.join(folder_path, f"{safe_title}.jpg")
                        
                        
                    
                            urllib.request.urlretrieve(img_url, img_save)
                        except:
                            print(f"이미지 다운로드 실패 ({title}): {e}")
                            img_save = "다운로드 실패"

                        
                search_list.append(keyword)
                title_list.append(title)
                name_list.append(name)
                price_list.append(price)
                com_list.append(com)
                date_list.append(ddatee)
                image_list.append(img_save)
            
            next_start_index = page + 1

            try:
                next_btn = wait.until(
                EC.element_to_be_clickable((By.LINK_TEXT, str(next_start_index)))
                
                )
                next_btn.click()
                time.sleep(1)

            except Exception as e:
                print("이동할 페이지 없어서 종료", e)
                break
            
    df = pd.DataFrame({
        "검색어": search_list,
        "도서명": title_list,
        "저자": name_list,
        "가격": price_list,
        "출판사": com_list,
        "출판일": date_list,
        "이미지 저장 경로": image_list
    })
    df.to_excel(f"yes24_검색결과.xlsx", index=False)

    driver.close()
    return 



In [67]:
yes24_crawl(3)

1번 페이지 크롤링
2번 페이지 크롤링
이동할 페이지 없어서 종료 Message: 
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff769923fa5+14925]
	chromedriver!GetHandleVerifier [0x7ff769924000+14980]
	chromedriver!(No symbol) [0x7ff76946793d]
	chromedriver!(No symbol) [0x7ff7694c1aad]
	chromedriver!(No symbol) [0x7ff7694c1dac]
	chromedriver!(No symbol) [0x7ff7695127d7]
	chromedriver!(No symbol) [0x7ff76950f39b]
	chromedriver!(No symbol) [0x7ff7694b401c]
	chromedriver!(No symbol) [0x7ff7694b4f43]
	chromedriver!GetHandleVerifier [0x7ff769f07591+5f7f11]
	chromedriver!GetHandleVerifier [0x7ff769f01902+5f2282]
	chromedriver!GetHandleVerifier [0x7ff769f27115+617a95]
	chromedriver!GetHandleVerifier [0x7ff769941dce+3274e]
	chromedriver!GetHandleVerifier [0x7ff76994a82c+3b1ac]
	chromedriver!GetHandleVerifier [0x7ff76992d744+1e0c4]
	chromedriver!GetHandleVerifier [0x7ff76992d8d4+1e254]
	chromedriver!GetHandleVerifier [0x7ff769911447+1dc7]
	KERNEL32!BaseThreadInitThunk [0x7ffcbc847614+14]
	ntdll!RtlUserThreadStart [0x7ffcbe342

In [ ]:
def kyono_crawl(max_page, keyword):
    pass

In [ ]:
def aladin_crawl(max_page, keyword):
    pass

In [11]:
def crawl_start(yes24, kyobo, aladin):
    keyword = input("검색어 입력: ")
    df01 = yes24_crawl(yes24, keyword)

    keyword = input("검색어 입력: ")
    df02 = kyono_crawl(kyobo)
    
    keyword = input("검색어 입력: ")
    df03 = aladin_crawl(aladin)

    # with pd.ExcelWriter("finential.xlsx") as writer:
    #     df01.to_excel(writer, sheet_name="yes24 시트", index=False)
    #     df02.to_excel(writer, sheet_name="kyobo 시트", index=False)
    #     df03.to_excel(writer, sheet_name="aladin 시트", index=False)


In [ ]:
crawl_start(3, 2, 4)